In [52]:
import pandas as pd 

In [53]:
%store -r df 
print(df.head())

  Invoice StockCode                          Description  Quantity  \
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1  536365     71053                  WHITE METAL LANTERN         6   
2  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  Price  Customer ID         Country  
0 2010-12-01 08:26:00   2.55        17850  United Kingdom  
1 2010-12-01 08:26:00   3.39        17850  United Kingdom  
2 2010-12-01 08:26:00   2.75        17850  United Kingdom  
3 2010-12-01 08:26:00   3.39        17850  United Kingdom  
4 2010-12-01 08:26:00   3.39        17850  United Kingdom  


In [54]:

df = df[~df['Invoice'].astype(str).str.startswith('C')]

* Remove transactions marked as cancellations (invoices starting with 'C')

In [55]:
df = df.rename(columns={'Customer ID': 'CustomerID'})
df = df.dropna(subset=['CustomerID'])

* Standardize column naming by removing spaces from the customer identifier
* Drop rows with missing customer identifiers to ensure validity for customer analytics

In [56]:
print(len(df))
print(df['CustomerID'].count())

397925
397925


* Verify total remaining row count and count of non-null customer IDs (both should match)


In [57]:
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

* Exclude invalid transactions by filtering out non-positive quantities and prices

In [58]:
df = df[df['Quantity'] <= 10000]
df = df[df['Price'] <= 1000]


* Restrain to ≤ 10,000 units per line item to isolate standard wholesale transactions from manual inventory adjustments.
*  Restrain to ≤ $1,000 per unit to eliminate systemic billing errors or banking adjustments from the retail product mix.

This optimization excluded only 21 extreme outlier rows. By preserving 99.9% of valid transaction data while eliminating severe anomalies (Max Quantity normalized to 4,800; Max Price normalized to $908.16).

In [59]:
df['Revenue'] = df['Quantity'] * df['Price']

* Calculate total revenue per transaction line item

In [60]:
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')


* Extract year-month period for time-series and trend analysis

In [61]:
df.to_csv('../data/cleaned_retail.csv', index=False)